# Kidney Lab Trend Analysis

Month-to-month trend review for `public.kidney_lab_result`.

In [ ]:
import math

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sqlalchemy import create_engine
from scipy.stats import linregress

plt.style.use('default')
pd.set_option('display.max_columns', 50)

# Database Connection

In [ ]:
DB_USER = "postgres"
DB_PASSWORD = "newpassword"
DB_HOST = "localhost"
DB_PORT = "5433"
DB_NAME = "DailyVitals"

connection_string = (
    f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}"
    f"@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

engine = create_engine(connection_string)

# Load Kidney Lab Data

In [ ]:
query = """
SELECT
    klr.kidney_lab_result_id,
    klr.person_id,
    p.first_name,
    p.last_name,
    klr.result_month,
    klr.albumin,
    klr.npcr,
    klr.potassium,
    klr.wktv,
    klr.calcium,
    klr.phosphorus,
    klr.ipth,
    klr.hemoglobin,
    klr.glucose,
    klr.cholesterol,
    klr.triglycerides,
    klr.bun,
    klr.creatinine,
    klr.notes
FROM public.kidney_lab_result klr
JOIN public.person p
    ON p.person_id = klr.person_id
ORDER BY klr.person_id, klr.result_month;
"""

df = pd.read_sql(query, engine)
df["result_month"] = pd.to_datetime(df["result_month"])
df["month"] = df["result_month"].dt.to_period("M").astype(str)
df["person_name"] = (df["first_name"] + " " + df["last_name"]).str.strip()

df.head()

# Select Person

In [ ]:
df[["person_id", "person_name"]].drop_duplicates().sort_values("person_name")

In [ ]:
# Set to a specific person_id if needed. Leave as None to use the first person in the result set.
PERSON_ID = None

if PERSON_ID is None:
    PERSON_ID = int(df["person_id"].iloc[0])

labs = (
    df[df["person_id"] == PERSON_ID]
    .sort_values("result_month")
    .reset_index(drop=True)
)

person_name = labs["person_name"].iloc[0]
print(f"Analyzing {person_name} ({len(labs)} monthly lab panels)")
labs

# Month-To-Month Changes

In [ ]:
metrics = [
    "albumin",
    "npcr",
    "potassium",
    "wktv",
    "calcium",
    "phosphorus",
    "ipth",
    "hemoglobin",
    "glucose",
    "cholesterol",
    "triglycerides",
    "bun",
    "creatinine",
]

display_names = {
    "albumin": "Albumin",
    "npcr": "nPCR",
    "potassium": "Potassium",
    "wktv": "wKt/V",
    "calcium": "Calcium",
    "phosphorus": "Phosphorus",
    "ipth": "iPTH",
    "hemoglobin": "Hemoglobin",
    "glucose": "Glucose",
    "cholesterol": "Cholesterol",
    "triglycerides": "Triglycerides",
    "bun": "BUN",
    "creatinine": "Creatinine",
}

changes = labs[["month", *metrics]].copy()
for metric in metrics:
    changes[f"{metric}_change"] = labs[metric].diff()
    changes[f"{metric}_pct_change"] = labs[metric].pct_change() * 100

change_columns = ["month"] + [f"{metric}_change" for metric in metrics]
changes[change_columns].tail(12)

# Latest Month Compared To Prior Month

In [ ]:
latest = labs.iloc[-1]
prior = labs.iloc[-2] if len(labs) > 1 else None

latest_summary_rows = []
for metric in metrics:
    current = latest[metric]
    previous = np.nan if prior is None else prior[metric]
    change = np.nan if prior is None else current - previous
    pct_change = np.nan if prior is None or previous == 0 else (change / previous) * 100
    latest_summary_rows.append({
        "Metric": display_names[metric],
        "Current": current,
        "Previous": previous,
        "Change": change,
        "Pct Change": pct_change,
    })

latest_summary = pd.DataFrame(latest_summary_rows)
latest_summary

# Trend Direction Model

In [ ]:
x = np.arange(len(labs))
trend_rows = []

for metric in metrics:
    y = labs[metric].astype(float).to_numpy()
    if len(y) >= 2:
        slope, intercept, r, p, std_err = linregress(x, y)
    else:
        slope, intercept, r, p, std_err = np.nan, np.nan, np.nan, np.nan, np.nan

    trend_rows.append({
        "Metric": display_names[metric],
        "Start": y[0],
        "Latest": y[-1],
        "Total Change": y[-1] - y[0],
        "Slope Per Month": slope,
        "R Value": r,
        "P Value": p,
    })

trend_summary = pd.DataFrame(trend_rows).sort_values("Metric")
trend_summary

# Key Lab Trends

In [ ]:
key_metrics = ["albumin", "npcr", "potassium", "wktv", "calcium", "phosphorus", "hemoglobin", "creatinine"]

fig, axes = plt.subplots(4, 2, figsize=(14, 12), sharex=True)
axes = axes.flatten()

for ax, metric in zip(axes, key_metrics):
    ax.plot(labs["result_month"], labs[metric], marker="o", linewidth=2)
    ax.set_title(display_names[metric])
    ax.grid(True, alpha=0.3)

for ax in axes[len(key_metrics):]:
    ax.axis("off")

fig.suptitle(f"Kidney Lab Trends - {person_name}", fontsize=16)
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

# All Lab Trends

In [ ]:
cols = 3
rows = math.ceil(len(metrics) / cols)
fig, axes = plt.subplots(rows, cols, figsize=(16, rows * 3.2), sharex=True)
axes = axes.flatten()

for ax, metric in zip(axes, metrics):
    ax.plot(labs["result_month"], labs[metric], marker="o", linewidth=2)
    ax.set_title(display_names[metric])
    ax.grid(True, alpha=0.3)

for ax in axes[len(metrics):]:
    ax.axis("off")

fig.suptitle(f"All Kidney Lab Trends - {person_name}", fontsize=16)
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

# Month-To-Month Delta Bars

In [ ]:
delta_metrics = ["albumin", "npcr", "potassium", "wktv", "calcium", "phosphorus", "hemoglobin", "creatinine"]

fig, axes = plt.subplots(4, 2, figsize=(14, 12), sharex=True)
axes = axes.flatten()

for ax, metric in zip(axes, delta_metrics):
    delta = labs[metric].diff()
    colors = np.where(delta >= 0, "seagreen", "crimson")
    ax.bar(labs["result_month"], delta, color=colors, width=20)
    ax.axhline(0, color="black", linewidth=1)
    ax.set_title(f"{display_names[metric]} Month-to-Month Change")
    ax.grid(True, axis="y", alpha=0.3)

for ax in axes[len(delta_metrics):]:
    ax.axis("off")

fig.autofmt_xdate()
plt.tight_layout()
plt.show()

# Percent Change Heatmap

In [ ]:
pct_change = labs.set_index("month")[metrics].pct_change() * 100
pct_change = pct_change.rename(columns=display_names)

fig, ax = plt.subplots(figsize=(14, max(4, len(pct_change) * 0.55)))
image = ax.imshow(pct_change.fillna(0), aspect="auto", cmap="RdYlGn")

ax.set_xticks(np.arange(len(pct_change.columns)))
ax.set_xticklabels(pct_change.columns, rotation=45, ha="right")
ax.set_yticks(np.arange(len(pct_change.index)))
ax.set_yticklabels(pct_change.index)

for i in range(len(pct_change.index)):
    for j in range(len(pct_change.columns)):
        value = pct_change.iloc[i, j]
        label = "" if pd.isna(value) else f"{value:.0f}%"
        ax.text(j, i, label, ha="center", va="center", fontsize=8)

ax.set_title(f"Month-to-Month Percent Change - {person_name}")
fig.colorbar(image, ax=ax, label="Percent change")
plt.tight_layout()
plt.show()

# Latest Narrative Summary

In [ ]:
latest_month = latest["result_month"].strftime("%b %Y")
prior_month = "n/a" if prior is None else prior["result_month"].strftime("%b %Y")

print(f"Kidney Lab Summary - {person_name}")
print()
print(f"Latest month: {latest_month}")
print(f"Compared against: {prior_month}")
print()

for _, row in latest_summary.iterrows():
    change = row["Change"]
    if pd.isna(change):
        print(f"{row['Metric']}: {row['Current']:.2f}")
    else:
        sign = "+" if change >= 0 else ""
        print(f"{row['Metric']}: {row['Current']:.2f} ({sign}{change:.2f} month over month)")